# Phase 3 — Step 3.3 (fixed): Retrain on CPU + TFLite Export

Single self-contained notebook.  Every cell imports everything it needs.

**Why this notebook exists:** the first training used GPU which silently switched the LSTM to CuDNN, an op that does not exist in TFLite.  This notebook retrains on CPU, then converts and downloads the TFLite files.

## Cell 1 — Imports (CuDNN disabled)

In [ ]:
import os
# Disable GPU so TensorFlow uses plain LSTM, not CuDNN
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import json
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs visible (should be 0): {len(gpus)}")

## Cell 2 — Mount Drive, load data, build splits

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/nsl-data'
MODEL_DIR  = '/content/drive/MyDrive/nsl-checkpoints'
TFLITE_DIR = '/content/drive/MyDrive/nsl-tflite'
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(TFLITE_DIR, exist_ok=True)

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))
with open(os.path.join(DATA_DIR, 'label_map.json')) as f:
    label_map = json.load(f)
n_classes = len(label_map)
print(f"X shape: {X.shape}")
print(f"Classes: {n_classes}")

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = {int(c): float(w)
                     for c, w in zip(np.unique(y_train), weights)}
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## Cell 3 — Build the model

In [ ]:
def build_model(input_shape, n_classes):
    inputs = keras.Input(shape=input_shape, name='landmarks')
    norm = layers.BatchNormalization(name='batch_norm')(inputs)
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.3),
        name='lstm_1')(norm)
    x = layers.Bidirectional(
        layers.LSTM(64, dropout=0.3), name='lstm_2')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation='softmax',
                           name='predictions')(x)
    return keras.Model(inputs=inputs, outputs=outputs, name='nsl_lstm_cpu')

model = build_model(input_shape=(30, 1662), n_classes=n_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Cell 4 — Train

In [ ]:
cbs = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model_cpu.h5'),
        monitor='val_loss', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=16,
    class_weight=class_weight_dict,
    callbacks=cbs, verbose=1,
)

## Cell 5 — Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"\nComparison vs GPU-trained: 0.7826 -- this run should be close.")

## Cell 6 — Convert to TFLite float32

In [ ]:
print("Converting to float32 TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float32 = converter.convert()
float32_path = os.path.join(TFLITE_DIR, 'nsl_model.tflite')
with open(float32_path, 'wb') as f:
    f.write(tflite_float32)
print(f"Saved {float32_path}")
print(f"Size: {len(tflite_float32)/1024:.1f} KB")

## Cell 7 — Convert to INT8 quantized TFLite

In [ ]:
print("Converting to INT8 TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

calibration_data = X[np.random.choice(len(X), 100, replace=False)
                    ].astype(np.float32)

def representative_dataset_gen():
    for sample in calibration_data:
        yield [sample[np.newaxis, :, :]]

converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()
int8_path = os.path.join(TFLITE_DIR, 'nsl_model_int8.tflite')
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)
print(f"Saved {int8_path}")
print(f"Size: {len(tflite_int8)/1024:.1f} KB")
print(f"\nSize reduction: "
      f"{(1 - len(tflite_int8)/len(tflite_float32))*100:.1f}%")

## Cell 8 — Quick TFLite accuracy check on the test set

In [ ]:
def run_tflite(interpreter, X, input_details, output_details, is_int8):
    preds = []
    in_scale, in_zp = input_details[0]['quantization']
    for i in range(len(X)):
        sample = X[i:i+1]
        if is_int8:
            sample = (sample / in_scale + in_zp).astype(np.int8)
        else:
            sample = sample.astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details[0]['index'])
        if is_int8:
            out_scale, out_zp = output_details[0]['quantization']
            out = (out.astype(np.float32) - out_zp) * out_scale
        preds.append(int(out[0].argmax()))
    return np.array(preds)

print("Verifying float32 TFLite...")
interp_f32 = tf.lite.Interpreter(model_path=float32_path, num_threads=2)
interp_f32.allocate_tensors()
in_f32  = interp_f32.get_input_details()
out_f32 = interp_f32.get_output_details()
preds_f32 = run_tflite(interp_f32, X_test, in_f32, out_f32, is_int8=False)
acc_f32 = (preds_f32 == y_test).mean()
print(f"  float32 TFLite test accuracy: {acc_f32:.4f}")

print("\nVerifying INT8 TFLite...")
interp_i8 = tf.lite.Interpreter(model_path=int8_path, num_threads=2)
interp_i8.allocate_tensors()
in_i8  = interp_i8.get_input_details()
out_i8 = interp_i8.get_output_details()
preds_i8 = run_tflite(interp_i8, X_test, in_i8, out_i8, is_int8=True)
acc_i8 = (preds_i8 == y_test).mean()
print(f"  INT8 TFLite test accuracy   : {acc_i8:.4f}")

print(f"\nAccuracy delta (float32 - INT8): {acc_f32 - acc_i8:+.4f}")
print(f"  {'GOOD -- INT8 close to float32' if abs(acc_f32 - acc_i8) < 0.03 else 'WATCH -- quantization cost is significant'}")

## Cell 9 — Download both TFLite files

In [ ]:
from google.colab import files
files.download(float32_path)
files.download(int8_path)
print("\nSave to your repo:")
print("  model/tflite/nsl_model.tflite")
print("  model/tflite/nsl_model_int8.tflite")
print("\nThen copy nsl_model_int8.tflite to:")
print("  flutter_app/assets/models/")